#Intialize notebook 

In [0]:
from Scripts.Silver_Layer_Scripts.helpers import *
deleteAll("data_analytics", "silverlayer", spark)

In [0]:
from Scripts.Silver_Layer_Scripts.ColumnMapConfig import *
from pyspark.sql import functions as F 
from pyspark.sql import Window

root = "data_analytics.silverlayer"

silver_layer_tables = spark.catalog.listTables(root)

column_mappings ={
    "customer_location": customer_location,
    "customer_demographic_info" : customer_demographic_info,
    "customer_information" : customer_information,
    "product_catagories": product_catagories,
    "product_information": product_information,
    "sales_details": sales_details
}

#Data transformations

##Table rename
loading tables from the bronze layer to silver layer, at the same time renaming them 

In [0]:
tables = spark.catalog.listTables("data_analytics.bronzelayer")

for table in tables:
    (
        spark.read.table(f"{table.catalog}.{table.namespace[0]}.{table.name}").na.fill("n/a")
        .write.mode("overwrite")
        .saveAsTable(f"{root}.{table_names[table.name]}")
    )

##Trim whitespaces in columns with string datatypes
At the same time replacing empty strings with **n/a**

In [0]:
for df_table in silver_layer_tables:

    column_list = spark.catalog.listColumns(f"{df_table.catalog}.{df_table.namespace[0]}.{df_table.name}")
    string_columns = [col.name for col in column_list if col.dataType == 'string']

    if len(string_columns) > 0:
        df = spark.read.table(f"{df_table.catalog}.{df_table.namespace[0]}.{df_table.name}")
        transformations = {
            col : F.when(F.trim(F.col(col)) == "", "n/a")
            .otherwise(F.trim(F.col(col))) 
            for col in string_columns
        }
        df = df.withColumns(transformations)
        df.write.mode("overwrite").saveAsTable(f"{df_table.catalog}.{df_table.namespace[0]}.{df_table.name}")
    

##Change column names
mapping names defined in the **ColumnMapConfig.py** script to silver tables

In [0]:
for table in silver_layer_tables:

    spark.sql(f"ALTER TABLE {table.catalog}.{table.namespace[0]}.{table.name} SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');")
     

    current_columns = spark.catalog.listColumns(f"{table.catalog}.{table.namespace[0]}.{table.name}")
    
    target_names = column_mappings.get(table.name)
    
    if not target_names:
        print(f"Warning: No mapping found for table {table.name}. Skipping.")
        continue

    column_name_list = [val.name for val in current_columns]   
    target_list = list(target_names.values()) if isinstance(target_names, dict) else list(target_names)
    
    for current_name, target_name in zip(column_name_list, target_list):
       
        if current_name != target_name:
            rename_query = f"ALTER TABLE {table.catalog}.{table.namespace[0]}.{table.name} RENAME COLUMN {current_name} TO {target_name};"
            try:
                spark.sql(rename_query)
                print(f"✅ Success: {table.name} ({current_name} -> {target_name})")
            except Exception as e:
                print(f"❌ Failed on table {table.name}: ({current_name} -> {target_name})")


##Customer table tranformations

###Customer information table

In [0]:
df = spark.table(f"{root}.customer_information")
#************quality check query******************
#hold = df.groupBy("Customer_id").count().filter("count > 1")
#DONE
#***************************************************

#************Cleanse dublicate Ids*************
df = df.withColumn(
    "Row_Number", F.row_number().over(Window.partitionBy("Customer_id").orderBy(F.desc("Date_created")))
    )
df = df.filter((df.Row_Number == 1) & (df.Date_created.isNotNull())).drop("Row_Number")
#***********************************************#


In [0]:
##DONE
df = (
    df.withColumn(
        "Gender",
        F.when(F.upper(df["Gender"]) == "M", "Male")
        .when(F.upper(df["Gender"]) == "F", "Female")
        .otherwise(df["Gender"]))
    .withColumn(
        "Marital_status",
        F.when(F.upper(df["Marital_status"]) == "M", "Married")
        .when(F.upper(df["Marital_status"]) == "S", "Single")
        .otherwise(df["Marital_status"])
    )
)
df.write.mode("overwrite").saveAsTable(f"{root}.customer_information")

###Customer Demographic information table


In [0]:
df = spark.table(f"data_analytics.silverlayer.customer_demographic_info")
df.groupby(["Customer_key"]).count().filter("count > 1").show() 

In [0]:
df = (df
      .withColumn(
    "Gender",
    F.when(F.upper(df["Gender"]) == "M", "Male")
    .when(F.upper(df["Gender"]) == "F", "Female")
    .otherwise(df["Gender"])
    )
)
df.write.mode("overwrite").saveAsTable(f"{root}.customer_demographic_info")

###Customer location table

In [0]:
df = spark.table(f"{root}.customer_location")

(
    df.withColumn(
        "Customer_id",
        F.replace("Customer_id", F.lit("-"), F.lit(""))
    )
).show()


##Product tables tranformations

###Product information table

In [0]:

##DONE
df = df.withColumn(
        "Catagory_id",
        F.trim(F.substring(df.Product_key, 1, 5))
    ).withColumn(
        "Product_key",
        F.trim(F.substring(df.Product_key, 7,F.length(df.Product_key)))
    )


In [0]:
#*********quality check query********
#df.filter((df.Cost < 0) | df.Cost.isNull() ).show()
#DONE 
#************************************

df = df.withColumn(
        "Cost",
        F.when((df.Cost < 0) | (df.Cost.isNull()), 0).otherwise(df.Cost)
    )
#display(df)

In [0]:
#***********quality check query********
#df = df.select("Product_key", "Start_date", "End_date").filter((df.Start_date > df.End_date) & (df.Product_key.isin(["HL-U509-R", "HL-U509"])))
#DONE
#*************Endl******************
df = df.withColumn(
    "End_date",
    F.when(
        df.Start_date > df.End_date, 
        F.lead(df.Start_date).over(Window.partitionBy(df.Product_key).orderBy(df.Start_date))-1
        )
    .otherwise(df.End_date)
)
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{root}.product_information")


###Product category table

In [0]:
df = spark.table(f"{root}.product_catagories")
df.show()

In [0]:
df.withColumn(
        "Product_key",
        F.replace("Product_key", F.lit("_"), F.lit("-"))
    ).show()

##Sales tables transformations

###Properly format the date
The sales_details table **('12345678' -> 'yyyy-mm-dd')** <br>
Invalid dates and empty entries are replaced with **1777-01-01** <br>
And finally cast the column to the proper datatype

In [0]:
spark.table(f"{root}.sales_details").display()

In [0]:
df = spark.read.table(f"{root}.sales_details")

sls_columns = ["Order_date", "Ship_date", "Due_date"]
transformations = {
    column :F.to_date(
        F.when(
            (F.col(column).isNull()) | (F.length(F.col(column)) != 8), 
            "1777-01-01"
        ).otherwise(F.col(column)), 
        "yyyyMMdd"
    )
    for column in sls_columns
}

df = df.withColumns(transformations)

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{root}.sales_details")